In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import comb

### Scatter graph

In [ ]:

# n=144 Hamming Bound
def get_hamming_bound(n: int):
    n = 144
    t_values = np.arange(0, 31)
    R_bound = []
    t_norm = []

    for t in t_values:
        sum_term = sum([comb(n, j, exact=True) * (3**j) for j in range(t + 1)])
        f2_t = (1 / n) * np.log2(float(sum_term))
        
        R = 1.0 - f2_t
        if R >= 0:
            R_bound.append(R)
            t_norm.append(t / n)
    
    return R_bound, t_norm


In [ ]:
# GB

# green markers: lambda = 0.5 
lambda_1_rate = [
    0.3333333333,
    0.2222222222,
    0.2152777778,
    0.2916666667,
]

lambda_1_ptnorm = [
    2.459,
    4.852,
    5.035,
    3.655,
]

# red markers: lambda = 1.0
lambda_05_rate = [
    0.09722222222,
    0.1111111111,
    0.1666666667,
    0.25,
    0.25,
    0.1666666667,
    0.2083333333,
    0.2222222222,
    0.2222222222,
    0.25,
    0.2777777778,
    0.2083333333,
]

lambda_05_ptnorm = [
    6.798,
    6.537,
    5.193,
    3.99,
    5.031,
    5.186,
    4.729,
    5.01,
    3.614,
    3.827,
    3.788,
    4.302,
]

lambda_1_ptnorm = np.array(lambda_1_ptnorm) / 144
lambda_05_ptnorm = np.array(lambda_05_ptnorm) / 144

# black marker: Gross code benchmark
gross_code = (12/144, 0.042)     # [[144, 12, 12]]


In [ ]:

n = 144
R_bound, t_norm = get_hamming_bound(n)

plt.figure(figsize=(8, 6))
plt.plot(R_bound, t_norm, '-', color='tab:blue', label=f'n={n} Hamming bound')
# plt.scatter(lambda_05_k_16[0], lambda_05_k_16[1], color='tab:green', marker='x', s=60, label='[[144, 16]]')
# plt.scatter(lambda_1_k_36[0], lambda_1_k_36[1], color='tab:red', marker='x', s=60, label='[[144, 36]]')

plt.scatter(lambda_05_rate, lambda_05_ptnorm, color='tab:green', marker='+', s=60, label='lambda = 0.5')
plt.scatter(lambda_1_rate, lambda_1_ptnorm, color='tab:red', marker='+', s=60, label='lambda = 1')

plt.scatter(gross_code[0], gross_code[1], color='black', marker='x', s=60, label='Gross code [[144, 12]]')

plt.xlabel('Code Rate $R = k/n$', fontsize=12)
plt.ylabel('Normalized pseudo-distance $\hat{t}/n$', fontsize=12)
plt.title('Operating Points of Generalised Bicycle Codes', fontsize=14)
# plt.title('Operating Points of Bivariate Bicycle Codes (found in distance mode)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)

plt.xlim(-0.05, 0.85)
plt.ylim(0.00, 0.21)

plt.tight_layout()
plt.show()

### Step graph

In [ ]:
import re

def extract_best_values(file_path):
    pattern = re.compile(r"\(Best value:\s*(-?\d+\.\d+)\)")
    best_values = []
    
    with open(file_path, 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                val = float(match.group(1))
                best_values.append(val)
    
    return  best_values


path = "./bayopLTEg12.e2728217.5.txt" 
y_data = extract_best_values(path)

x_data = [i for i in range(0, (len(y_data) * 2), 2)]

print(f"Extracted {len(y_data)} data points.")
print("First 5 X-coordinates:", x_data[:7])
print("First 5 Y-coordinates:", y_data[:7])


In [ ]:
import os

def process_all_runs(folder_path):
    all_runs = []
    
    for file_name in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file_name)
        run_data = extract_best_values(file_path)
        
        if len(run_data) == 101:
            all_runs.append(run_data)
        elif len(run_data) == 200:
            all_runs.append(run_data[::2] + [run_data[-1]])
        else:
            print(f"{file_name} has {len(run_data)} points instead of 100, skipping")

    if not all_runs:
        print("no files found")
        return

    all_runs_matrix = np.array(all_runs)
    mean_line = np.mean(all_runs_matrix, axis=0)
    std_line = np.std(all_runs_matrix, axis=0)

    return mean_line, std_line


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.set_ylim(-0.62, -0.18)
ax.set_xticks(np.arange(0, 201, 50))

folders = {
    "./logs/g12": "GB-BO mean: k=12, density=6",
    "./logs/g2": "GB-BO mean: varied params",
    "./logs/l1_12_6": "BB-BO mean",
    "./logs/RS_gb": "GB-RS mean",
}

for folder, descr in folders.items():
    mean, std = process_all_runs(folder)

    ax.step(x_data, mean, label=descr, where='post')

    ax.fill_between(x_data, mean - std, mean + std, 
                     alpha=0.2, step='post')

ax.set_xlabel('Evaluation index', fontsize=12)
ax.set_ylabel('Best-so-far objective', fontsize=12)
ax.legend(loc='lower right', fontsize=10, framealpha=0.4)
plt.show()